# Đánh giá Hybrid và Reranker trên Colab

Phần Người 1 Team 2: chạy VLRA-54/55 và xuất bằng chứng cho VLRA-58.
Model và index nằm trên máy Colab; máy cá nhân chỉ mở trình duyệt.
Không cần Ollama, API key, Java, Docker hay mount Google Drive.

Chọn **Runtime → Change runtime type → GPU**, rồi chạy từng ô từ trên xuống.
[Colab miễn phí không bảo đảm GPU hoặc thời lượng phiên](https://research.google.com/colaboratory/faq.html#resource-limits).
Nếu chưa được cấp GPU, dừng và thử lại sau; notebook không mua tài nguyên.

Mặc định dùng 40 câu Golden cũ ở chế độ **exploratory**, chưa phải benchmark
Dev/Test chính thức. Với Dev đã review, đổi cấu hình ở ô 3 trước khi chạy tiếp.
Runner giữ top 20, RRF k=60, batch rerank 2 và giới hạn 512 token/cặp.
Không nới timeout hoặc cắt passage để ép kết quả thành công.


## 1. Lấy code và cài thư viện


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Banh-Van-Hiep/vietnamese-legal-rag.git"
BRANCH = "feature/team2-rag"
REPO = Path("/content/vietnamese-legal-rag")
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
else:
    # Giữ checkout đã có; không pull hoặc xóa thay đổi khi chạy lại ô.
    origin = subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip()
    if origin != REPO_URL:
        raise RuntimeError("Thư mục hiện có không thuộc repo này; hãy dùng runtime mới.")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
HEAD = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
if subprocess.check_output(["git", "status", "--porcelain"], text=True).strip():
    raise RuntimeError("Checkout có thay đổi chưa commit; hãy kiểm tra trước khi đo.")
print("Code:", HEAD)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[test,retrieval,dense,reranker]"], check=True)


## 2. Kiểm tra GPU


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Chưa có CUDA GPU. Chọn GPU trong Runtime → Change runtime type, rồi kết nối lại.")
DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name())
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)


## 3. Chọn dữ liệu và ghim revision model

Giữ `MODE = "exploratory"` để khảo sát Golden cũ. Khi có Dev đã review và
đóng băng, tải file lên vùng Files của Colab, đặt `MODE = "dev"` và sửa
`GOLDEN` thành đường dẫn file đó. Mẫu Dev cần `split: "dev"` và
`relevant_unit_groups` theo hướng dẫn trong repo. Không dùng Test để chọn cấu hình.

Hai revision để trống sẽ được tra một lần từ Hugging Face rồi ghi vào bằng
chứng. Lần tái lập sau, dán lại các SHA đã ghi; không tự thay model giữa hai nhánh.


In [ ]:
import hashlib
import json
from collections import Counter
from datetime import datetime, timezone
from huggingface_hub import model_info
from scripts.eval_reranker import validate_dataset

MODE = "exploratory"
GOLDEN = REPO / "team2_rag/data/golden/golden_phase1_30cai.json"
CHUNKS = REPO / "data/doanh-nghiep/processed/chunks.jsonl"
EMBEDDING_MODEL = "AITeamVN/Vietnamese_Embedding"
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"
EMBEDDING_REVISION = ""  # Điền SHA cũ khi cần tái lập.
RERANKER_REVISION = ""

if MODE not in ("exploratory", "dev"):
    raise ValueError("MODE phải là exploratory hoặc dev.")
cases = json.loads(GOLDEN.read_text(encoding="utf-8"))
chunks = [json.loads(line) for line in CHUNKS.read_text(encoding="utf-8").splitlines() if line.strip()]
corpus = {row["chunk_id"]: row for row in chunks}
if len(corpus) != len(chunks):
    raise ValueError("Corpus có chunk_id trùng.")
validate_dataset(cases, corpus, exploratory=MODE == "exploratory")
EMBEDDING_REVISION = model_info(EMBEDDING_MODEL, revision=EMBEDDING_REVISION or "main").sha
RERANKER_REVISION = model_info(RERANKER_MODEL, revision=RERANKER_REVISION or "main").sha
CORPUS_HASH = hashlib.sha256(CHUNKS.read_bytes()).hexdigest()
VERSION = f"colab-{CORPUS_HASH[:12]}-{EMBEDDING_REVISION[:8]}"
RUN = Path("/content/legal-rag-runs") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
EVIDENCE = RUN / "evidence"
EVIDENCE.mkdir(parents=True, exist_ok=False)
BM25_INDEX, DENSE_INDEX, QDRANT = [RUN / name for name in ("bm25", "dense", "qdrant")]
settings = dict(mode=MODE, head=HEAD, corpus_sha256=CORPUS_HASH,
                golden_sha256=hashlib.sha256(GOLDEN.read_bytes()).hexdigest(),
                index_version=VERSION, embedding_model=EMBEDDING_MODEL,
                embedding_revision=EMBEDDING_REVISION, reranker_revision=RERANKER_REVISION,
                device=DEVICE, gpu=torch.cuda.get_device_name(), torch=torch.__version__, cuda=torch.version.cuda)
(EVIDENCE / "run-settings.json").write_text(json.dumps(settings, ensure_ascii=False, indent=2), encoding="utf-8")
(EVIDENCE / "golden.json").write_bytes(GOLDEN.read_bytes())
(EVIDENCE / "packages.txt").write_bytes(subprocess.check_output([sys.executable, "-m", "pip", "freeze"]))
print("Số câu:", len(cases), dict(Counter(c["expected_status"] for c in cases)))
print("Corpus:", len(chunks), "chunks; chế độ:", MODE)
print("Revision embedding:", EMBEDDING_REVISION, "reranker:", RERANKER_REVISION)
print("Bằng chứng:", EVIDENCE)


## 4. Dựng index trên Colab

Dùng trực tiếp thư viện Team 1, giữ nguyên chunk và thuật toán. Cả hai index
mang cùng version và tập IDs. Đây là index mới để thực nghiệm từ corpus của
commit đã ghi; không mặc định là snapshot MVP trước đây.

Ô này tải embedding model về Colab. Khi hoàn tất, nó đóng Qdrant và giải phóng
model trước khi runner nạp lại. Nếu build dở, chạy lại từ ô 3 để tạo thư mục
run mới; không xóa hoặc ghi đè index đang có.


In [ ]:
import gc
import shutil
import time
from qdrant_client import QdrantClient
from team1_data.retrieval import BM25Retriever, DenseRetriever, SentenceTransformerEmbedder

if any(path.exists() for path in (BM25_INDEX, DENSE_INDEX, QDRANT)):
    raise RuntimeError("Run này đã có index. Chạy lại từ ô 3 để tạo run mới.")
start = time.perf_counter()
sparse = BM25Retriever.from_jsonl(CHUNKS)
if set(sparse.chunk_ids) != set(corpus):
    raise ValueError("Bộ lọc hiệu lực làm thay đổi tập chunk; cần đối chiếu snapshot trước khi đo.")
sparse.save(BM25_INDEX, index_version=VERSION)
embedder = SentenceTransformerEmbedder(name=EMBEDDING_MODEL, revision=EMBEDDING_REVISION,
                                       batch_size=8, device=DEVICE)
client = QdrantClient(path=str(QDRANT))
try:
    dense = DenseRetriever.from_jsonl(CHUNKS, embedder=embedder, client=client, index_version=VERSION)
    if set(dense.chunk_ids) != set(sparse.chunk_ids):
        raise ValueError("Hai index không cùng tập chunk IDs.")
    dense.save(DENSE_INDEX)
finally:
    client.close()
for name, path in (("bm25", BM25_INDEX), ("dense", DENSE_INDEX)):
    shutil.copy2(path / "index_manifest.json", EVIDENCE / f"{name}-manifest.json")
(EVIDENCE / "build-time.json").write_text(json.dumps(dict(elapsed_ms=(time.perf_counter() - start) * 1000)))
del sparse, dense, embedder, client
gc.collect()
torch.cuda.empty_cache()
print("Đã dựng index:", VERSION)


## 5. Chạy so sánh và đọc báo cáo

Hai nhánh dùng chung Hybrid top 20. Reranker chạy strict, không fallback.
Exit 0 nghĩa là chạy đủ; 1 có lỗi ở từng câu; 2 bị chặn khi chuẩn bị.
Cặp dài hơn 512 token được ghi là lỗi, không tự truncate. Run có lỗi vẫn giữ
báo cáo để phân tích, không dùng xác nhận đạt chỉ tiêu.

Latency là wall time trên Colab, gồm chuyển tensor và chép điểm về CPU để
đợi CUDA hoàn tất. Nó chưa phải latency Mac, LLM hoặc E2E của sản phẩm.


In [ ]:
from IPython.display import Markdown, display

REPORT = EVIDENCE / f"reranker-{MODE}.json"
command = [sys.executable, "-m", "scripts.eval_reranker",
           "--golden", str(GOLDEN), "--chunks", str(CHUNKS),
           "--bm25-index", str(BM25_INDEX), "--dense-index", str(DENSE_INDEX),
           "--qdrant-path", str(QDRANT), "--device", DEVICE, "--embedding-device", DEVICE,
           "--reranker-revision", RERANKER_REVISION, "--output", str(REPORT)]
if MODE == "exploratory":
    command.append("--exploratory")
# Không raise ngay: giữ quyền đọc và tải báo cáo của run bị lỗi.
result = subprocess.run(command, cwd=REPO)
print("Exit code:", result.returncode)
if REPORT.exists():
    report = json.loads(REPORT.read_text(encoding="utf-8"))
    print("Trạng thái:", report["status"])
    display(Markdown(REPORT.with_suffix(".md").read_text(encoding="utf-8")))
else:
    print("Chưa có báo cáo. Kiểm tra lỗi phía trên rồi chạy lại ô này.")


## 6. Tải bằng chứng về máy

ZIP chứa báo cáo, raw candidates/scores, Golden đầu vào, manifest index,
revision model và danh sách thư viện. Không đóng gói weights hoặc vector
index lớn. Tải ngay khi kết thúc vì filesystem Colab có thể mất khi phiên dừng.

Dùng raw cases để phân tích VLRA-55. Với chế độ exploratory, ghi đúng phạm vi
khảo sát; chưa chốt cấu hình VLRA-58 hoặc tuyên bố đạt benchmark chính thức.


In [ ]:
from google.colab import files

archive = shutil.make_archive(str(RUN / "reranker-results"), "zip", root_dir=EVIDENCE)
print("Tải bằng chứng:", archive)
files.download(archive)
